In [29]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer

In [30]:
# Full tables (optional peek); pipeline reloads in the next cell.
news_labels_full = pd.read_csv("news_topics.csv")
news_train_full = pd.read_csv("news.csv")
news_labels_full.shape, news_train_full.shape

((2606875, 3), (23149, 2))

In [31]:
N = 5000
news_train = pd.read_csv("news.csv")
news_labels = pd.read_csv("news_topics.csv")
news_labels["cat"] = news_labels["cat"].str.strip()

articles_5k = news_train.head(N)
ids_5k = set(articles_5k["id"])

# TF-IDF — text is already stemmed / cleaned, just split on whitespace
tfidf_vec = TfidfVectorizer()
X_tfidf = tfidf_vec.fit_transform(articles_5k["article"])
print(f"X_tfidf: {X_tfidf.shape}")

# One-hot labels: one row per article, columns = all unique categories
labels_sub = news_labels[news_labels["id"].isin(ids_5k)]
cat_lists = labels_sub.groupby("id", sort=False)["cat"].apply(list)
labels_5k = articles_5k[["id"]].merge(cat_lists.rename("cats"), on="id", how="left")
labels_5k["cats"] = labels_5k["cats"].apply(lambda x: x if isinstance(x, list) else [])

mlb = MultiLabelBinarizer()
Y_labels = mlb.fit_transform(labels_5k["cats"])
print(f"Y_labels: {Y_labels.shape}  ({len(mlb.classes_)} unique categories)")

Y_labels

X_tfidf: (5000, 21871)
Y_labels: (5000, 95)  (95 unique categories)


array([[0, 0, 0, ..., 0, 0, 1],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [32]:
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# --- h1 labels only (CCAT, ECAT, GCAT, MCAT) ---
H1_TOPICS = ["CCAT", "ECAT", "GCAT", "MCAT"]
labels_h1 = news_labels[
    (news_labels["id"].isin(ids_5k)) & (news_labels["cat"].isin(H1_TOPICS))
]
cat_lists_h1 = labels_h1.groupby("id", sort=False)["cat"].apply(list)
aligned = articles_5k[["id"]].merge(cat_lists_h1.rename("cats"), on="id", how="left")
aligned["cats"] = aligned["cats"].apply(lambda x: x if isinstance(x, list) else [])

mlb_h1 = MultiLabelBinarizer(classes=H1_TOPICS)
Y = mlb_h1.fit_transform(aligned["cats"])
print(f"Y: {Y.shape}  counts={dict(zip(H1_TOPICS, Y.sum(axis=0)))}")

# --- PCA via TruncatedSVD ---
svd = TruncatedSVD(n_components=300, random_state=42)
X = svd.fit_transform(X_tfidf)
print(f"PCA: {X_tfidf.shape} → {X.shape}, var={svd.explained_variance_ratio_.sum():.3f}")

# --- 5-fold CV with LinearSVC ---
K = 5
kf = KFold(n_splits=K, shuffle=True, random_state=42)
topic_metrics = {t: {"accuracy":[], "precision":[], "recall":[], "f1":[]} for t in H1_TOPICS}
fold_cm = {t: np.zeros((2,2), dtype=int) for t in H1_TOPICS}

for fold, (tr, te) in enumerate(kf.split(X), 1):
    clf = OneVsRestClassifier(LinearSVC(max_iter=5000, random_state=42))
    clf.fit(X[tr], Y[tr])
    Yp = clf.predict(X[te])
    for i, t in enumerate(H1_TOPICS):
        yt, yp = Y[te][:,i], Yp[:,i]
        topic_metrics[t]["accuracy"].append(accuracy_score(yt, yp))
        topic_metrics[t]["precision"].append(precision_score(yt, yp, zero_division=0))
        topic_metrics[t]["recall"].append(recall_score(yt, yp, zero_division=0))
        topic_metrics[t]["f1"].append(f1_score(yt, yp, zero_division=0))
        fold_cm[t] += confusion_matrix(yt, yp, labels=[0,1])
    print(f"  Fold {fold}/{K} done")

# --- Results ---
rows = []
for t in H1_TOPICS:
    m = topic_metrics[t]
    rows.append({"Topic": t, "Accuracy": np.mean(m["accuracy"]),
                 "Precision": np.mean(m["precision"]),
                 "Recall": np.mean(m["recall"]), "F1": np.mean(m["f1"])})
results_df = pd.DataFrame(rows)
print("\n=== SVM Baseline — H1 Topics (5-fold CV) ===\n")
print(results_df.to_string(index=False))
print(f"\nMacro-avg F1: {results_df['F1'].mean():.4f}")

print("\n=== Confusion Matrices (summed over all folds) ===")
for t in H1_TOPICS:
    print(f"\n{t}:")
    print(pd.DataFrame(fold_cm[t], index=["Actual 0","Actual 1"], columns=["Pred 0","Pred 1"]))

Y: (5000, 4)  counts={'CCAT': 2471, 'ECAT': 826, 'GCAT': 1291, 'MCAT': 1277}
PCA: (5000, 21871) → (5000, 300), var=0.438
  Fold 1/5 done
  Fold 2/5 done
  Fold 3/5 done
  Fold 4/5 done
  Fold 5/5 done

=== SVM Baseline — H1 Topics (5-fold CV) ===

Topic  Accuracy  Precision   Recall       F1
 CCAT    0.9208   0.918423 0.921678 0.919879
 ECAT    0.9304   0.862994 0.689690 0.765999
 GCAT    0.9510   0.939832 0.864908 0.900703
 MCAT    0.9532   0.936523 0.876437 0.905425

Macro-avg F1: 0.8730

=== Confusion Matrices (summed over all folds) ===

CCAT:
          Pred 0  Pred 1
Actual 0    2326     203
Actual 1     193    2278

ECAT:
          Pred 0  Pred 1
Actual 0    4082      92
Actual 1     256     570

GCAT:
          Pred 0  Pred 1
Actual 0    3638      71
Actual 1     174    1117

MCAT:
          Pred 0  Pred 1
Actual 0    3647      76
Actual 1     158    1119


In [33]:
# ---------- Out-of-sample evaluation on news_test.csv ----------
news_test = pd.read_csv("news_test.csv")
print(f"Test articles: {news_test.shape[0]}")

# TF-IDF transform (same vocabulary as training)
X_test_raw = tfidf_vec.transform(news_test["article"])
X_test = svd.transform(X_test_raw)

# True labels for test set (h1 only)
ids_test = set(news_test["id"])
labels_test_sub = news_labels[
    (news_labels["id"].isin(ids_test)) & (news_labels["cat"].isin(H1_TOPICS))
]
cat_lists_test = labels_test_sub.groupby("id", sort=False)["cat"].apply(list)
aligned_test = news_test[["id"]].merge(cat_lists_test.rename("cats"), on="id", how="left")
aligned_test["cats"] = aligned_test["cats"].apply(lambda x: x if isinstance(x, list) else [])
Y_test = mlb_h1.transform(aligned_test["cats"])

# Train on ALL 5k, predict on test
clf_final = OneVsRestClassifier(LinearSVC(max_iter=5000, random_state=42))
clf_final.fit(X, Y)
Y_pred_test = clf_final.predict(X_test)

# Results
rows_oos = []
for i, t in enumerate(H1_TOPICS):
    yt, yp = Y_test[:, i], Y_pred_test[:, i]
    rows_oos.append({
        "Topic": t,
        "Accuracy": accuracy_score(yt, yp),
        "Precision": precision_score(yt, yp, zero_division=0),
        "Recall": recall_score(yt, yp, zero_division=0),
        "F1": f1_score(yt, yp, zero_division=0),
    })
oos_df = pd.DataFrame(rows_oos)
print("\n=== SVM Baseline — H1 Topics (OOS on news_test) ===\n")
print(oos_df.to_string(index=False))
print(f"\nMacro-avg F1: {oos_df['F1'].mean():.4f}")

print("\n=== Confusion Matrices (OOS) ===")
for i, t in enumerate(H1_TOPICS):
    cm = confusion_matrix(Y_test[:, i], Y_pred_test[:, i], labels=[0, 1])
    print(f"\n{t}:")
    print(pd.DataFrame(cm, index=["Actual 0","Actual 1"], columns=["Pred 0","Pred 1"]))

Test articles: 390616

=== SVM Baseline — H1 Topics (OOS on news_test) ===

Topic  Accuracy  Precision   Recall       F1
 CCAT  0.908729   0.914957 0.890168 0.902392
 ECAT  0.924645   0.856584 0.594125 0.701613
 GCAT  0.944073   0.929677 0.878332 0.903276
 MCAT  0.944588   0.925806 0.850853 0.886748

Macro-avg F1: 0.8485

=== Confusion Matrices (OOS) ===

CCAT:
          Pred 0  Pred 1
Actual 0  190161   15318
Actual 1   20334  164803

ECAT:
          Pred 0  Pred 1
Actual 0  326575    5794
Actual 1   23641   34606

GCAT:
          Pred 0  Pred 1
Actual 0  266764    7716
Actual 1   14130  102006

MCAT:
          Pred 0  Pred 1
Actual 0  284232    6791
Actual 1   14854   84739
